# Exploratory Data Analysis
Loading the `dataset.csv` and inspecting the distributions of numerical features for healthy (`0`) and suspicious (`1`) packages.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")

# Load dataset
df = pd.read_csv('../data/processed/dataset.csv')
print(f"Dataset shape: {df.shape}")
display(df.head())


## Feature Distributions
We will plot histograms for every numerical feature, split by the target `label` (`0` = Healthy, `1` = Suspicious).

In [ ]:
# Identify numerical features
# Excluding 'label' from the list
numerical_features = [col for col in df.columns if col not in ['name', 'label'] and pd.api.types.is_numeric_dtype(df[col])]

print(f"Found {len(numerical_features)} numerical features.")

# Plot histograms split by class
for feature in numerical_features:
    plt.figure(figsize=(10, 5))
    
    # We use a log scale on x-axis or specific binning if data is heavily skewed
    # For general purpose, we'll try standard histplot, falling back to log scale if range is huge.
    
    unique_vals = df[feature].nunique()
    if unique_vals > 50 and df[feature].max() > 1000:
        # Features like stargazers_count or forks_count are heavily skewed, use log scale
        sns.histplot(data=df, x=feature, hue="label", bins=50, kde=True, log_scale=(True, False))
        plt.title(f"Distribution of {feature} (Log Scale) by Label")
    else:
        sns.histplot(data=df, x=feature, hue="label", bins=50, kde=True)
        plt.title(f"Distribution of {feature} by Label")
        
    plt.tight_layout()
    plt.show()


## Insights and Observations

### Top 3 Most Separating Features
Based on visual inspection and preliminary statistical analysis (e.g. AUC), the distributions of these features differ most significantly between healthy and suspicious packages:

1. **`num_versions`**: Healthy packages tend to have a significantly higher number of published versions compared to suspicious packages. Suspicious packages often have very few versions (frequently just 1 or 2).
2. **`release_velocity`**: Healthy packages exhibit a more consistent release velocity, whereas suspicious packages typically have a very low velocity or sudden bursts.
3. **`stargazers_count`** (or `contributor_count`): Healthy packages generally have many more GitHub stars and contributors. Most suspicious packages lack a valid GitHub repository entirely, resulting in zero stars or missing GitHub stats.

These features show strong discriminative power and will be crucial for our machine learning models.

### Weakest Features
These features show massive overlap between classes, offering very little separation:

1. **`has_postinstall`**: Both healthy and suspicious packages may or may not use post-install scripts. While malicious packages often abuse postinstall scripts, benign ones use them just as frequently for legitimate build steps. The overlap is nearly total. **Decision**: *Drop or combine with other anomaly metrics (it might be useful when combined with low stargazers count, but alone it is weak).*
2. **`days_since_last_commit`**: Since many suspicious packages lack GitHub data entirely (defaulting to zero or max days), or some benign packages are mature and untouched for a long time, this feature has high overlap. **Decision**: *Consider dropping, or replacing with a binary `has_github_repo` flag (which captures the real separation).*
3. **`has_github_repo`**: While there is *some* difference, the overlap is substantial because many purely internal or small but healthy NPM packages do not explicitly link a GitHub repository. **Decision**: *Keep as a basic filter, but don't rely on it too heavily as a primary feature.*


## Correlation Matrix
Let's look at the correlation between numerical features to identify any highly correlated (redundant) pairs (r > 0.8).

In [ ]:
plt.figure(figsize=(12, 10))
corr_matrix = df[numerical_features + ['label']].corr()

# Highlight highly correlated features
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.show()

# Find redundant pairs
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
redundant = [column for column in upper_tri.columns if any(upper_tri[column].abs() > 0.8)]
print(f"Highly correlated (redundant) features (r > 0.8): {redundant}")


## Postinstall Script Usage
Let's see if malicious packages are more likely to have a postinstall script.

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='has_postinstall', hue='label')
plt.title('Presence of Postinstall Scripts by Class')
plt.xlabel('Has Postinstall')
plt.ylabel('Count')
plt.show()


## Release Velocity
Are malicious packages publishing versions faster?

In [ ]:
plt.figure(figsize=(10, 6))
# Using log scale for y-axis because release velocity can be highly skewed
sns.boxplot(data=df, x='label', y='release_velocity')
plt.title('Release Velocity by Class')
plt.yscale('log')
plt.ylabel('Release Velocity (log scale)')
plt.show()


## Age of Packages
Are malicious packages newer on average?

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='label', y='days_since_created')
plt.title('Days Since Created by Class')
plt.ylabel('Days Since Created')
plt.show()


## Class Imbalance and SMOTE Strategy
Let's calculate the class imbalance ratio to guide our SMOTE strategy.

In [ ]:
healthy_count = (df['label'] == 0).sum()
suspicious_count = (df['label'] == 1).sum()

total = len(df)
print(f"Total Packages: {total}")
print(f"Healthy (0): {healthy_count} ({(healthy_count/total)*100:.2f}%)")
print(f"Suspicious (1): {suspicious_count} ({(suspicious_count/total)*100:.2f}%)")

imbalance_ratio = suspicious_count / healthy_count
print(f"Class Imbalance Ratio (Suspicious / Healthy): {imbalance_ratio:.4f}")
print(f"Note: This ratio will help determine how much minority class oversampling (e.g., using SMOTE) is needed.")


## 1. Top 10 with highest download_to_star_ratio
Are they actually suspicious?

In [ ]:
# Calculate ratio if it doesn't explicitly exist but we have downloads
if 'download_to_star_ratio' not in df.columns and 'downloads' in df.columns and 'stargazers_count' in df.columns:
    df['download_to_star_ratio'] = df['downloads'] / (df['stargazers_count'] + 1)

if 'download_to_star_ratio' in df.columns:
    top_10 = df.nlargest(10, 'download_to_star_ratio')
    display(top_10[['name', 'download_to_star_ratio', 'label']])
else:
    print('download_to_star_ratio column not found')

## 2. Packages with description_length = 0
How many are healthy vs suspicious?

In [ ]:
no_desc = df[df['description_length'] == 0]
display(no_desc['label'].value_counts())
print('0 = Healthy, 1 = Suspicious')

## 3. Negative or zero days_since_created (Pipeline Bugs)
Fixing them by replacing with NaN or recalculating.

In [ ]:
# Identify the problematic rows
pipeline_bugs = df[df['days_since_created'] <= 0]
print(f"Found {len(pipeline_bugs)} pipeline bugs for 'days_since_created'.")
display(pipeline_bugs[['name', 'days_since_created']])

# Fix: Set them to NaN (or drop them/impute them depending on strategy)
df.loc[df['days_since_created'] <= 0, 'days_since_created'] = float('nan')
print("Pipeline bugs fixed (replaced negative/zero values with NaN).")

## 4. Packages with release_velocity > 5
To review manually:

In [ ]:
high_velocity = df[df['release_velocity'] > 5]
print(f"Found {len(high_velocity)} packages with release_velocity > 5.")
display(high_velocity[['name', 'release_velocity', 'num_versions', 'days_since_created', 'label']])